In [1]:
# Configuration et imports
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import pdfplumber
import fitz  # PyMuPDF
import time
import re
import json
from collections import defaultdict
from pprint import pprint
from datetime import datetime
import random

# Configuration
PDF_DIR = Path('../data/pdfs')
print(f"📁 Répertoire PDF: {PDF_DIR.absolute()}")

📁 Répertoire PDF: /home/vincheetah/Documents/Programmation/Python/PyVolley/notebooks/../data/pdfs


In [2]:
# Charger les parsers existants
from pyvolley.parsers.v2 import MatchSheetParserV2
from pyvolley.parsers.v3 import MatchSheetParserV3
from pyvolley.parsers.base import ParseResult

parser_v2 = MatchSheetParserV2()
parser_v3 = MatchSheetParserV3()

print(f"Parser V2: {parser_v2.name} v{parser_v2.version}")
print(f"Parser V3: {parser_v3.name} v{parser_v3.version}")

Parser V2: MatchSheetParserV2 v2.1.0
Parser V3: MatchSheetParserV3 v3.0.0


In [3]:
# Collecter un échantillon de PDFs depuis différentes compétitions/saisons
def collect_sample_pdfs(base_dir: Path, sample_size: int = 50, seed: int = 42) -> list[Path]:
    """Collecte un échantillon représentatif de PDFs."""
    all_pdfs = list(base_dir.glob('**/*.pdf'))
    print(f"Total PDFs trouvés: {len(all_pdfs)}")
    
    random.seed(seed)
    sample = random.sample(all_pdfs, min(sample_size, len(all_pdfs)))
    
    # Afficher la distribution
    seasons = defaultdict(int)
    for pdf in sample:
        parts = pdf.parts
        for p in parts:
            if re.match(r'\d{4}-\d{4}', p):
                seasons[p] += 1
                break
    
    print(f"\n🎯 Échantillon: {len(sample)} PDFs")
    print("Distribution par saison:")
    for season, count in sorted(seasons.items()):
        print(f"  {season}: {count}")
    
    return sample

sample_pdfs = collect_sample_pdfs(PDF_DIR, sample_size=100)

Total PDFs trouvés: 104219

🎯 Échantillon: 100 PDFs
Distribution par saison:
  2020-2021: 1
  2022-2023: 13
  2023-2024: 27
  2024-2025: 34
  2025-2026: 25


In [4]:
# Évaluation comparative des parsers
def evaluate_parsers(pdfs: list[Path], parsers: dict) -> dict:
    """Compare les performances de plusieurs parsers."""
    results = {name: {
        'success': 0, 'failure': 0, 'warnings': 0,
        'total_time_ms': 0, 'errors': [],
        'field_completeness': defaultdict(int),
        'parsed_matches': []
    } for name in parsers}
    
    for i, pdf_path in enumerate(pdfs):
        if (i + 1) % 20 == 0:
            print(f"Progress: {i+1}/{len(pdfs)}")
        
        for name, parser in parsers.items():
            try:
                result = parser.parse(pdf_path)
                
                results[name]['total_time_ms'] += result.parse_time_ms
                
                if result.success:
                    results[name]['success'] += 1
                    if result.match:
                        m = result.match
                        results[name]['parsed_matches'].append({
                            'file': pdf_path.name,
                            'code': m.code_match,
                            'equipe_a': m.equipe_a.nom if m.equipe_a else None,
                            'equipe_b': m.equipe_b.nom if m.equipe_b else None,
                            'score': m.score_final,
                            'vainqueur': m.vainqueur_nom,
                            'sets': [(s.score_a, s.score_b) for s in (m.sets or [])],
                            'joueurs_a': len(m.equipe_a.joueurs) if m.equipe_a else 0,
                            'joueurs_b': len(m.equipe_b.joueurs) if m.equipe_b else 0,
                        })
                        
                        # Comptage completeness
                        results[name]['field_completeness']['code_match'] += 1 if m.code_match else 0
                        results[name]['field_completeness']['equipe_a'] += 1 if m.equipe_a and m.equipe_a.nom else 0
                        results[name]['field_completeness']['equipe_b'] += 1 if m.equipe_b and m.equipe_b.nom else 0
                        results[name]['field_completeness']['score'] += 1 if m.score_final else 0
                        results[name]['field_completeness']['vainqueur'] += 1 if m.vainqueur_nom else 0
                        results[name]['field_completeness']['sets'] += 1 if m.sets else 0
                        results[name]['field_completeness']['joueurs'] += 1 if (m.equipe_a and m.equipe_a.joueurs) else 0
                        results[name]['field_completeness']['date'] += 1 if m.date else 0
                        results[name]['field_completeness']['lieu'] += 1 if m.lieu else 0
                        results[name]['field_completeness']['arbitres'] += 1 if m.arbitres else 0
                else:
                    results[name]['failure'] += 1
                
                if result.warnings:
                    results[name]['warnings'] += len(result.warnings)
                    
                if result.errors:
                    results[name]['errors'].append({
                        'file': pdf_path.name,
                        'errors': result.errors
                    })
                    
            except Exception as e:
                results[name]['failure'] += 1
                results[name]['errors'].append({
                    'file': pdf_path.name,
                    'errors': [str(e)]
                })
    
    return results

# Exécuter l'évaluation
eval_results = evaluate_parsers(sample_pdfs, {
    'V2 (PyMuPDF)': parser_v2,
    'V3 (pdfplumber)': parser_v3
})

Progress: 20/100
Progress: 40/100
Progress: 60/100
Progress: 80/100
Progress: 100/100


In [5]:
# Afficher le résumé de l'évaluation
print("=" * 70)
print("RÉSUMÉ DE L'ÉVALUATION DES PARSERS")
print("=" * 70)

for name, data in eval_results.items():
    total = data['success'] + data['failure']
    success_rate = data['success'] / total * 100 if total > 0 else 0
    avg_time = data['total_time_ms'] / total if total > 0 else 0
    
    print(f"\n📊 {name}")
    print(f"   Succès: {data['success']}/{total} ({success_rate:.1f}%)")
    print(f"   Temps moyen: {avg_time:.2f} ms")
    print(f"   Warnings: {data['warnings']}")
    
    print(f"\n   Complétude des champs (sur {data['success']} matchs réussis):")
    for field, count in sorted(data['field_completeness'].items()):
        pct = count / data['success'] * 100 if data['success'] > 0 else 0
        bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
        print(f"      {field:15} {bar} {pct:5.1f}%")

RÉSUMÉ DE L'ÉVALUATION DES PARSERS

📊 V2 (PyMuPDF)
   Succès: 0/100 (0.0%)
   Temps moyen: 15.42 ms
   Warnings: 47

   Complétude des champs (sur 0 matchs réussis):

📊 V3 (pdfplumber)
   Succès: 100/100 (100.0%)
   Temps moyen: 414.66 ms
   Warnings: 0

   Complétude des champs (sur 100 matchs réussis):
      arbitres        █████░░░░░░░░░░░░░░░  29.0%
      code_match      ████████████████████ 100.0%
      date            ████████████████████ 100.0%
      equipe_a        ████████████████████ 100.0%
      equipe_b        ████████████████████ 100.0%
      joueurs         ░░░░░░░░░░░░░░░░░░░░   0.0%
      lieu            ████████████████████ 100.0%
      score           ████████████████████ 100.0%
      sets            ░░░░░░░░░░░░░░░░░░░░   0.0%
      vainqueur       ████████████████████ 100.0%


In [6]:
# Examiner quelques erreurs
print("\n" + "=" * 70)
print("ÉCHANTILLON D'ERREURS")
print("=" * 70)

for name, data in eval_results.items():
    print(f"\n{name} - Premières erreurs:")
    for err in data['errors'][:3]:
        print(f"  📄 {err['file']}")
        for e in err['errors'][:2]:
            print(f"     ❌ {e[:100]}..." if len(e) > 100 else f"     ❌ {e}")


ÉCHANTILLON D'ERREURS

V2 (PyMuPDF) - Premières erreurs:
  📄 LICA_RMAR048.pdf
     ❌ Erreur de parsing: 1 validation error for Equipe
nom
  String should have at least 2 characters [typ...
  📄 PTRA69_JF6A015.pdf
     ❌ Erreur de parsing: 1 validation error for Equipe
nom
  String should have at least 2 characters [typ...
  📄 PTIDF91_1F4055.pdf
     ❌ Erreur de parsing: 1 validation error for Equipe
nom
  String should have at least 2 characters [typ...

V3 (pdfplumber) - Premières erreurs:


In [7]:
# Analyser un PDF en détail pour comprendre le format
def analyze_pdf_structure(pdf_path: Path):
    """Analyse détaillée de la structure d'un PDF."""
    print(f"\n{'='*70}")
    print(f"ANALYSE DE: {pdf_path.name}")
    print('='*70)
    
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[0]
        words = page.extract_words()
        tables = page.extract_tables()
        full_text = page.extract_text() or ""
        
        print(f"\nDimensions: {page.width} x {page.height}")
        print(f"Nombre de mots: {len(words)}")
        print(f"Nombre de tables: {len(tables)}")
        
        # Zones verticales
        zones = defaultdict(list)
        for w in words:
            y_zone = int(w['top'] // 50) * 50
            zones[y_zone].append(w)
        
        print("\n--- ZONES VERTICALES ---")
        for y in sorted(zones.keys()):
            texts = [w['text'] for w in sorted(zones[y], key=lambda x: x['x0'])[:15]]
            print(f"Y={y:3d}: {' | '.join(texts)}")
        
        print("\n--- TABLES ---")
        for i, table in enumerate(tables):
            print(f"\nTable {i+1} ({len(table)} lignes):")
            for row in table[:5]:
                print(f"  {row}")
            if len(table) > 5:
                print(f"  ... ({len(table) - 5} lignes de plus)")
    
    return words, tables, full_text

# Analyser le premier PDF de l'échantillon
test_pdf = sample_pdfs[0]
words, tables, full_text = analyze_pdf_structure(test_pdf)


ANALYSE DE: LICA_RMAR048.pdf

Dimensions: 841.89 x 595.28
Nombre de mots: 361
Nombre de tables: 5

--- ZONES VERTICALES ---
Y=  0: RMA | Ville: | Salle: | ANTIBES | BERTONE | - | CHAMPIONNAT | REGIONAL | MASCULIN | POULE | A | SENIOR | | | MASCULIN | Samedi
Y= 50: Ligue | PROVENCE-ALPES-CÔTE | Formation | Ordre | de | de | Service | Départ | D’AZUR | S | E | I | II | III | IV
Y=100: Remplaçants | Tours | au | service | 1 | 2 | 3 | Joueur | Score | 5 | 6 | 7 | N° | T | 1
Y=150: Formation | Ordre | 4 | de | Joueur | de | Service | Départ | 8 | N° | S | E | T | I | II
Y=200: Remplaçants | Tours | au | service | 1 | 2 | 3 | Score | 5 | 6 | 7 | 3 | T | T | 4
Y=250: S | E | T | I | II | III | 4 | IV | 8 | V | VI | I | II | III | IV
Y=300: 5 | T | T | T | 04 | 05 | 06 | 07 | 10 | 13 | MIROCHKIN | SAINT | BAHGAT | BERZA | DORBANI
Y=350: A | SANCTIONS | P | E | D | A/B | EQU.A | DEMANDE | Set | NON | EQU.B | Score | FONDEE | REMARQUES | 15
Y=400: Arbitres | 1er | LEBONNOIS | NOAM | NOM | Préno

In [8]:
# Afficher le texte complet d'un PDF pour comprendre le format
print("\n--- TEXTE COMPLET ---\n")
print(full_text[:3000])


--- TEXTE COMPLET ---

RMA - CHAMPIONNAT REGIONAL MASCULIN POULE A Match: RMAR048 - Jour: 10
Ville: ANTIBES Samedi 16 Décembre 2023 à 18h00
Salle: BERTONE SENIOR | MASCULIN
Ligue PROVENCE-ALPES-CÔTE D’AZUR OLYMPIQUES ANTIBES JUAN PINS NICE VOLLEY-BALL
S Début: Fin: S Début: Fin:
Ordre de Service E I II III IV V VI I II III IV V VI E I II III IV V VI I II III IV V VI
Formation de Départ
Joueur N° T T
Remplaçants
Score
1 5 1 2
2 6 T T T T
Tours au service
3 7
4 8
S Début: Fin: S Début: Fin:
Ordre de Service E I II III IV V VI I II III IV V VI E I II III IV V VI I II III IV V VI
Formation de Départ
Joueur N° T T
Remplaçants
Score
1 5 3 4
2 6 T T T T
Tours au service
3 7
4 8
S OLYMPIQUES ANTIBES JUAN PINS NICE VOLLEY-BALL
E I II III IV V VI I II III IV V VI I II III IV V VI N° Nom Prénom Licence N° Nom Prénom Licence
02 TRIMOREAU MATHIS 2367719 03 MARACHE ROBIN 2185723
T 03 AVRIL LUCAS 2165784 04 FREI SEBASTIEN 2081595
04 MIROCHKIN KYRSAN 2272061 09 KECHICHE MOHAMED 2463348
05 SAINT GRATI

## Analyse des Problèmes Identifiés

Basé sur l'évaluation, notons les problèmes identifiés avec les parsers actuels.

In [9]:
# Comparer les résultats de parsing entre V2 et V3 pour le même fichier
def compare_parse_results(pdf_path: Path):
    """Compare les résultats des deux parsers pour un même PDF."""
    print(f"\n{'='*70}")
    print(f"COMPARAISON: {pdf_path.name}")
    print('='*70)
    
    r2 = parser_v2.parse(pdf_path)
    r3 = parser_v3.parse(pdf_path)
    
    def show_match(label, result):
        print(f"\n--- {label} ---")
        print(f"Succès: {result.success}")
        print(f"Temps: {result.parse_time_ms:.2f}ms")
        if result.errors:
            print(f"Erreurs: {result.errors}")
        if result.warnings:
            print(f"Warnings: {result.warnings}")
        
        if result.match:
            m = result.match
            print(f"Code: {m.code_match}")
            print(f"Equipe A: {m.equipe_a.nom if m.equipe_a else 'N/A'}")
            print(f"Equipe B: {m.equipe_b.nom if m.equipe_b else 'N/A'}")
            print(f"Score: {m.score_final}")
            print(f"Vainqueur: {m.vainqueur_nom}")
            print(f"Sets: {[(s.score_a, s.score_b) for s in m.sets]}")
            print(f"Joueurs A: {len(m.equipe_a.joueurs) if m.equipe_a else 0}")
            print(f"Joueurs B: {len(m.equipe_b.joueurs) if m.equipe_b else 0}")
            print(f"Arbitres: {len(m.arbitres)}")
            print(f"Date: {m.date}")
            print(f"Lieu: {m.lieu}")
    
    show_match('V2 (PyMuPDF)', r2)
    show_match('V3 (pdfplumber)', r3)

# Comparer pour quelques PDFs
for pdf in sample_pdfs[:3]:
    compare_parse_results(pdf)


COMPARAISON: LICA_RMAR048.pdf

--- V2 (PyMuPDF) ---
Succès: False
Temps: 9.87ms
Erreurs: ["Erreur de parsing: 1 validation error for Equipe\nnom\n  String should have at least 2 characters [type=string_too_short, input_value='', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.10/v/string_too_short"]
Warnings: ['Match non joué']

--- V3 (pdfplumber) ---
Succès: True
Temps: 366.22ms
Code: RMAR048
Equipe A: OLYMPIQUES ANTIBES JUAN PINS
Equipe B: NICE VOLLEY-BALL
Score: 0/0
Vainqueur: NICE VOLLEY-BALL
Sets: []
Joueurs A: 0
Joueurs B: 4
Arbitres: 1
Date: 2023-12-16
Lieu: ANTIBES S

COMPARAISON: PTRA69_JF6A015.pdf

--- V2 (PyMuPDF) ---
Succès: False
Temps: 6.81ms
Erreurs: ["Erreur de parsing: 1 validation error for Equipe\nnom\n  String should have at least 2 characters [type=string_too_short, input_value='S', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.10/v/string_too_short"]

--- V3 (pdfplumber) ---
Succès: True
Temps: 

In [10]:
# Tester le nouveau parser V4
# Importer le module à nouveau pour récupérer les changements
import importlib
import pyvolley.parsers.v4
importlib.reload(pyvolley.parsers.v4)

from pyvolley.parsers.v4 import MatchSheetParserV4

parser_v4 = MatchSheetParserV4()
print(f"Parser V4: {parser_v4.name} v{parser_v4.version}")

# Test sur quelques PDFs
print("\n=== TEST DU PARSER V4 ===")
for pdf in sample_pdfs[:5]:
    result = parser_v4.parse(pdf)
    print(f"\n📄 {pdf.name}")
    print(f"   Succès: {'✓' if result.success else '✗'}")
    print(f"   Temps: {result.parse_time_ms:.2f}ms")
    if result.errors:
        print(f"   Erreurs: {result.errors[0][:100]}...")
    if result.match:
        m = result.match
        print(f"   Code: {m.code_match}")
        print(f"   Équipes: {m.equipe_a.nom} vs {m.equipe_b.nom}")
        print(f"   Score: {m.score_final}")
        print(f"   Joueurs A: {len(m.equipe_a.joueurs)}")
        print(f"   Joueurs B: {len(m.equipe_b.joueurs)}")
        print(f"   Arbitres: {len(m.arbitres)}")

Parser V4: MatchSheetParserV4 v4.0.0

=== TEST DU PARSER V4 ===

📄 LICA_RMAR048.pdf
   Succès: ✓
   Temps: 357.37ms
   Code: RMAR048
   Équipes: OLYMPIQUES ANTIBES JUAN PINS vs NICE VOLLEY-BALL
   Score: 0/0
   Joueurs A: 10
   Joueurs B: 8
   Arbitres: 0

📄 PTRA69_JF6A015.pdf
   Succès: ✓
   Temps: 397.56ms
   Code: JF6A015
   Équipes: FIDESIENNE VB 2 vs ASV GARON
   Score: 3/0
   Joueurs A: 11
   Joueurs B: 11
   Arbitres: 0

📄 PTIDF91_1F4055.pdf
   Succès: ✓
   Temps: 326.09ms
   Code: 1F4055
   Équipes: UNION SPORTIVE WISSOUS VB vs VALLEE DE CHEVREUSE VOLLEY-BAL
   Score: 2/0
   Joueurs A: 5
   Joueurs B: 3
   Arbitres: 0

📄 ABCCS_3FE093.pdf
   Succès: ✓
   Temps: 366.43ms
   Code: 3FE093
   Équipes: COUDOUX VELAUX LA FARE VOLLEY- vs NIMES VOLLEY-BALL
   Score: 0/0
   Joueurs A: 10
   Joueurs B: 9
   Arbitres: 0

📄 PTPR13_CMA002.pdf
   Succès: ✓
   Temps: 389.51ms
   Code: CMA002
   Équipes: SALON VOLLEY vs BOUC BEL AIR VOLLEY
   Score: 2/0
   Joueurs A: 7
   Joueurs B: 10
   Arbit

In [11]:
# Évaluation complète du parser V4 vs V3
print("=== ÉVALUATION COMPARATIVE V3 vs V4 ===\n")

eval_v4_results = evaluate_parsers(sample_pdfs, {
    'V3 (pdfplumber)': parser_v3,
    'V4 (optimisé)': parser_v4
})

=== ÉVALUATION COMPARATIVE V3 vs V4 ===

Progress: 20/100
Progress: 40/100
Progress: 60/100
Progress: 80/100
Progress: 100/100


In [12]:
# Afficher les résultats de l'évaluation V3 vs V4
print("=" * 70)
print("RÉSUMÉ DE L'ÉVALUATION V3 vs V4")
print("=" * 70)

for name, data in eval_v4_results.items():
    total = data['success'] + data['failure']
    success_rate = data['success'] / total * 100 if total > 0 else 0
    avg_time = data['total_time_ms'] / total if total > 0 else 0
    
    print(f"\n📊 {name}")
    print(f"   Succès: {data['success']}/{total} ({success_rate:.1f}%)")
    print(f"   Temps moyen: {avg_time:.2f} ms")
    print(f"   Warnings: {data['warnings']}")
    
    print(f"\n   Complétude des champs (sur {data['success']} matchs réussis):")
    for field, count in sorted(data['field_completeness'].items()):
        pct = count / data['success'] * 100 if data['success'] > 0 else 0
        bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
        print(f"      {field:15} {bar} {pct:5.1f}%")

RÉSUMÉ DE L'ÉVALUATION V3 vs V4

📊 V3 (pdfplumber)
   Succès: 100/100 (100.0%)
   Temps moyen: 417.84 ms
   Warnings: 0

   Complétude des champs (sur 100 matchs réussis):
      arbitres        █████░░░░░░░░░░░░░░░  29.0%
      code_match      ████████████████████ 100.0%
      date            ████████████████████ 100.0%
      equipe_a        ████████████████████ 100.0%
      equipe_b        ████████████████████ 100.0%
      joueurs         ░░░░░░░░░░░░░░░░░░░░   0.0%
      lieu            ████████████████████ 100.0%
      score           ████████████████████ 100.0%
      sets            ░░░░░░░░░░░░░░░░░░░░   0.0%
      vainqueur       ████████████████████ 100.0%

📊 V4 (optimisé)
   Succès: 100/100 (100.0%)
   Temps moyen: 420.48 ms
   Warnings: 44

   Complétude des champs (sur 100 matchs réussis):
      arbitres        ░░░░░░░░░░░░░░░░░░░░   0.0%
      code_match      ████████████████████ 100.0%
      date            ████████████████████ 100.0%
      equipe_a        █████████████████

In [13]:
# Recharger et tester le parser V4 amélioré
import importlib
import pyvolley.parsers.v4
importlib.reload(pyvolley.parsers.v4)
from pyvolley.parsers.v4 import MatchSheetParserV4

parser_v4_new = MatchSheetParserV4()
print("=== TEST PARSER V4 AMÉLIORÉ ===\n")

# Test rapide sur quelques PDFs
test_results = []
for pdf in sample_pdfs[:10]:
    result = parser_v4_new.parse(pdf)
    test_results.append({
        'file': pdf.name,
        'success': result.success,
        'code': result.match.code_match if result.match else None,
        'vainqueur': result.match.vainqueur_nom if result.match else None,
        'score': result.match.score_final if result.match else None,
        'joueurs_a': len(result.match.equipe_a.joueurs) if result.match and result.match.equipe_a else 0,
        'joueurs_b': len(result.match.equipe_b.joueurs) if result.match and result.match.equipe_b else 0,
        'arbitres': len(result.match.arbitres) if result.match else 0,
        'lieu': result.match.lieu if result.match else None,
    })
    
for r in test_results:
    print(f"📄 {r['file'][:30]:30} | {r['code']:12} | Score: {r['score'] or 'N/A':5} | Vainqueur: {(r['vainqueur'] or 'N/A')[:25]:25} | J:{r['joueurs_a']:2}+{r['joueurs_b']:2} | A:{r['arbitres']} | Lieu: {(r['lieu'] or 'N/A')[:20]}")

=== TEST PARSER V4 AMÉLIORÉ ===

📄 LICA_RMAR048.pdf               | RMAR048      | Score: 0/0   | Vainqueur: NICE VOLLEY-BALL          | J:10+ 8 | A:0 | Lieu: ANTIBES S
📄 PTRA69_JF6A015.pdf             | JF6A015      | Score: 3/0   | Vainqueur: ASV GARON                 | J:11+11 | A:0 | Lieu: STE FOY LES LYON S
📄 PTIDF91_1F4055.pdf             | 1F4055       | Score: 2/0   | Vainqueur: UNION SPORTIVE WISSOUS    | J: 5+ 3 | A:0 | Lieu: WISSOUS S
📄 ABCCS_3FE093.pdf               | 3FE093       | Score: 0/0   | Vainqueur: COUDOUX VELAUX LA FARE    | J:10+ 9 | A:0 | Lieu: VELAUX D
📄 PTPR13_CMA002.pdf              | CMA002       | Score: 2/0   | Vainqueur: BOUC BEL AIR VOLLEY       | J: 7+10 | A:0 | Lieu: N/A
📄 LICA_PFBR063.pdf               | PFBR063      | Score: 3/1   | Vainqueur: ISTRES PROVENCE VOLLEY    | J: 9+12 | A:0 | Lieu: ISTRES S
📄 PTFL59_BMC004.pdf              | BMC004       | Score: 2/0   | Vainqueur: CAMBRAI 1                 | J: 4+ 4 | A:0 | Lieu: CAMBRAI S
📄 ABCCS_2MB065

In [14]:
# Analyser la zone des arbitres en détail
def analyze_arbitres_zone(pdf_path):
    """Analyse la zone des arbitres d'un PDF."""
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[0]
        full_text = page.extract_text() or ""
        
        # Afficher les lignes contenant des infos sur les arbitres
        print(f"\n=== {pdf_path.name} ===")
        print("\n--- Lignes contenant '1er', '2ème', 'Marqueur' ---")
        for line in full_text.split('\n'):
            if any(kw in line for kw in ['1er', '2ème', 'Marqueur', 'Arbitre']):
                print(f"  {line}")
        
        # Chercher dans les tables
        tables = page.extract_tables()
        for i, table in enumerate(tables):
            for row in table or []:
                if row:
                    row_text = ' '.join(str(c) for c in row if c)
                    if any(kw in row_text for kw in ['1er', '2ème', 'Marqueur', 'Arbitre']):
                        print(f"\n  Table {i}: {row}")

# Analyser quelques PDFs
for pdf in sample_pdfs[:3]:
    analyze_arbitres_zone(pdf)


=== LICA_RMAR048.pdf ===

--- Lignes contenant '1er', '2ème', 'Marqueur' ---
  Arbitres NOM Prénom Ligue Licence Signature Equipe A Equipe B OFFICIELS
  1er LEBONNOIS NOAM PAC 2367380 T R G P Durée par Set P G R T EA DEMIROVIC ASIM 1567002 EA SAUZEAU THOMAS 2357067
  2ème Début Fin Durée
  MarqueurINZIRILLO MAEVA PAC 2001951 Vainqueur: NICE VOLLEY-BALL 0/0 SIGNATURES

  Table 0: [None, None, None, None, None, None, None, None, None, None, None, None, None, 'Arbitres', None, None, 'NOM Prénom', None, None, None, None, None, None, None, None, None, 'Ligue', None, None, None, 'Licence', None, None, 'Signature', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]

  Table 0: [None, None, None, None, None, None, None, None, None, None, None, None, None, '1er', None, None, 'LEBONNOIS NOAM', None, None, None, None, None, None, None, None, None, 'PAC', None, None, None, '2367380', None, None, '', None, None, None, None, None, None, None, None, None, None,

In [15]:
# Test final du parser V4 amélioré
import importlib
import pyvolley.parsers.v4
importlib.reload(pyvolley.parsers.v4)
from pyvolley.parsers.v4 import MatchSheetParserV4

parser_v4_final = MatchSheetParserV4()
print("=== ÉVALUATION FINALE PARSER V4 ===\n")

# Évaluation complète
eval_v4_final = evaluate_parsers(sample_pdfs, {
    'V4 Final': parser_v4_final
})

=== ÉVALUATION FINALE PARSER V4 ===

Progress: 20/100
Progress: 40/100
Progress: 60/100
Progress: 80/100
Progress: 100/100


In [16]:
# Afficher les résultats finaux
print("=" * 70)
print("RÉSULTAT FINAL - PARSER V4")
print("=" * 70)

for name, data in eval_v4_final.items():
    total = data['success'] + data['failure']
    success_rate = data['success'] / total * 100 if total > 0 else 0
    avg_time = data['total_time_ms'] / total if total > 0 else 0
    
    print(f"\n📊 {name}")
    print(f"   Succès: {data['success']}/{total} ({success_rate:.1f}%)")
    print(f"   Temps moyen: {avg_time:.2f} ms")
    print(f"   Warnings: {data['warnings']}")
    
    print(f"\n   Complétude des champs (sur {data['success']} matchs réussis):")
    for field, count in sorted(data['field_completeness'].items()):
        pct = count / data['success'] * 100 if data['success'] > 0 else 0
        bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
        print(f"      {field:15} {bar} {pct:5.1f}%")

RÉSULTAT FINAL - PARSER V4

📊 V4 Final
   Succès: 100/100 (100.0%)
   Temps moyen: 402.53 ms
   Warnings: 41

   Complétude des champs (sur 100 matchs réussis):
      arbitres        █████████████████░░░  89.0%
      code_match      ████████████████████ 100.0%
      date            ████████████████████ 100.0%
      equipe_a        ████████████████████ 100.0%
      equipe_b        ████████████████████ 100.0%
      joueurs         ████████████████████ 100.0%
      lieu            █████████████████░░░  88.0%
      score           ████████████████████ 100.0%
      sets            ███████████░░░░░░░░░  59.0%
      vainqueur       ███████████████████░  99.0%
